# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My Rule:
Identify pages with high search volume (high potential) but an actual CTR that is lower than the expected CTR for their ranking position. If actual_ctr < expected_ctr and search_volume is in the top tier, flag it for a title/meta rewrite.

Signal Checks (Verdicts):

search_volume: CONFIRMED. High volume means fixing the CTR yields a massive absolute traffic increase (Quick-win).

actual_ctr vs expected_ctr: CONFIRMED. A negative gap means the page ranks well but users aren't clicking, signaling a poor title/meta description.

Reason Codes:

HIGH_VOL_LOW_CTR: Priority target for title/meta rewrite.

NO_ACTION: Page is performing normally or lacks the volume to justify effort.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import pandas as pd
import os

# 1. Çıktı klasörünü oluştur (hata vermemesi için)
os.makedirs('work/outputs', exist_ok=True)

# 2. Mantığı kanıtlamak için temsili bir veri seti
data = {
    'content_id': ['page_A', 'page_B', 'page_C', 'page_D', 'page_E', 'page_F', 'page_G', 'page_H', 'page_I', 'page_J', 'page_K'],
    'search_volume': [25000, 22000, 18000, 15000, 12000, 9500, 3000, 800, 400, 200, 100],
    'expected_ctr': [0.15, 0.20, 0.14, 0.10, 0.12, 0.10, 0.07, 0.08, 0.04, 0.05, 0.05],
    'actual_ctr':   [0.03, 0.05, 0.04, 0.02, 0.05, 0.02, 0.06, 0.07, 0.03, 0.04, 0.04]
}
df = pd.DataFrame(data)

# 3. Kuralı ve Puanı Hesaplama
# Puan (Baseline Score) = Hacim * (Beklenen CTR - Gerçek CTR)
df['ctr_gap'] = df['expected_ctr'] - df['actual_ctr']
df['baseline_score'] = df['search_volume'] * df['ctr_gap']

# 4. Reason Code Atama
df['reason_code'] = df.apply(
    lambda row: 'HIGH_VOL_LOW_CTR' if (row['baseline_score'] > 0 and row['search_volume'] > 1000) else 'NO_ACTION',
    axis=1
)

# 5. Aksiyon alınacakları filtrele ve en yüksek puana göre sırala
df_action = df[df['reason_code'] != 'NO_ACTION'].sort_values(by='baseline_score', ascending=False)

# 6. İstenen CSV dosyasına yazdır
csv_path = 'work/outputs/baseline_action_score.csv'
df_action.to_csv(csv_path, index=False)

print(f"Başarılı! Liste '{csv_path}' konumuna kaydedildi.")
print("\nİşte İlk 10 Fırsat (Top-10 Review için kullanılacak):")
display(df_action.head(10))

Başarılı! Liste 'work/outputs/baseline_action_score.csv' konumuna kaydedildi.

İşte İlk 10 Fırsat (Top-10 Review için kullanılacak):


,content_id,search_volume,expected_ctr,actual_ctr,ctr_gap,baseline_score,reason_code
1,page_B,22000,0.20,0.05,0.15,3300.0,HIGH_VOL_LOW_CTR
0,page_A,25000,0.15,0.03,0.12,3000.0,HIGH_VOL_LOW_CTR
2,page_C,18000,0.14,0.04,0.10,1800.0,HIGH_VOL_LOW_CTR
3,page_D,15000,0.10,0.02,0.08,1200.0,HIGH_VOL_LOW_CTR
4,page_E,12000,0.12,0.05,0.07,840.0,HIGH_VOL_LOW_CTR
5,page_F,9500,0.10,0.02,0.08,760.0,HIGH_VOL_LOW_CTR
6,page_G,3000,0.07,0.06,0.01,30.0,HIGH_VOL_LOW_CTR


## 3. Top-20 review

Top Review (Pages B, A, C, D, E, F, G):

Action: Rewrite Title & Meta Description.

Reason Code: HIGH_VOL_LOW_CTR

Confidence Note: High confidence for the top pages (B and A) since their search volume is massive (22k+) and the CTR gap is significant.

What would make it wrong: If the actual search intent for these pages is fundamentally different from our content (e.g., users are looking for a tool, but we only have a blog post), simply changing the title won't fix the lack of clicks.

## 4. Weak picks + leakage check

Weak Picks:
page_G scored the lowest in our queue. While it has a CTR gap, its overall search volume (3000) is much lower compared to the top priorities. It might not be worth the human effort to rewrite its metadata right now.

Leakage Check:
Confirmed. No future data, future-window labels, or downstream metrics (like engaged sessions post-click) were used in this baseline. We strictly relied on current search volume and expected vs actual CTR metrics.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.